In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

In [2]:
def count_tags(tags):
    counter = Counter(tags)
    total = sum(counter.values())

    df = pd.DataFrame(
        counter.items(),
        columns=["tag", "count"]
    )

    df["percent"] = df["count"] / total * 100
    df["entity"] = df["tag"].apply(
        lambda x: x.split("-")[1] if "-" in x else "O"
    )

    df = df.sort_values("count", ascending=False).reset_index(drop=True)
    return df


def count_tags_jsonl(path):
    counter = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            counter.update(item["tags"])

    total = sum(counter.values())

    df = pd.DataFrame(
        counter.items(),
        columns=["tag", "count"]
    )

    df["percent"] = df["count"] / total * 100
    df["entity"] = df["tag"].apply(
        lambda x: x.split("-")[1] if "-" in x else "O"
    )

    df = df.sort_values("count", ascending=False).reset_index(drop=True)
    return df

In [3]:
train_tags = count_tags_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_train_vifinner.jsonl")
train_tags

,tag,count,percent,entity
0,O,900059,80.927531,O
1,I-ORG,45554,4.095923,ORG
2,B-ORG,39631,3.563365,ORG
3,I-MONEY,25186,2.264564,MONEY
4,I-DATE,20663,1.857884,DATE
5,B-DATE,20437,1.837564,DATE
6,B-MONEY,13362,1.201425,MONEY
7,B-RATE,11932,1.072849,RATE
8,I-PERSON,8281,0.744574,PERSON
9,B-ASSET,7044,0.633351,ASSET


In [4]:
dev_tags = count_tags_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_dev_vifinner.jsonl")
dev_tags

,tag,count,percent,entity
0,O,192905,81.196149,O
1,I-ORG,9347,3.934270,ORG
2,B-ORG,8378,3.526406,ORG
3,I-MONEY,5294,2.228311,MONEY
4,I-DATE,4524,1.904209,DATE
5,B-DATE,4426,1.862959,DATE
6,B-MONEY,2824,1.188657,MONEY
7,B-RATE,2579,1.085534,RATE
8,I-PERSON,1710,0.719761,PERSON
9,I-VOLUME,1406,0.591803,VOLUME


In [5]:
test_tags = count_tags_jsonl("/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/final_test_vifinner.jsonl")
test_tags

,tag,count,percent,entity
0,O,192079,80.870261,O
1,I-ORG,9832,4.139528,ORG
2,B-ORG,8475,3.568196,ORG
3,I-MONEY,5446,2.292908,MONEY
4,I-DATE,4431,1.865566,DATE
5,B-DATE,4399,1.852094,DATE
6,B-MONEY,2892,1.217607,MONEY
7,B-RATE,2471,1.040355,RATE
8,I-PERSON,1675,0.705219,PERSON
9,B-ASSET,1576,0.663537,ASSET


In [9]:
def sentence_level_eda(jsonl_path):
    sentence_lengths = []
    entity_sentence_count = 0
    no_entity_sentence_count = 0
    entity_counter = Counter()
    entity_per_sentence = []
    bad_sentences = []

    total_sentences = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception as e:
                bad_sentences.append((idx, "JSON parse error"))
                continue

            tokens = obj.get("tokens")
            tags = obj.get("tags")

            if not tokens or not tags or len(tokens) != len(tags):
                bad_sentences.append((idx, "tokens-tags mismatch"))
                continue

            total_sentences += 1
            sent_len = len(tokens)
            sentence_lengths.append(sent_len)

            # Đếm entity trong câu
            entities_in_sentence = 0
            has_entity = False

            for tag in tags:
                if tag.startswith("B-"):
                    ent_type = tag[2:]
                    entity_counter[ent_type] += 1
                    entities_in_sentence += 1
                    has_entity = True

            entity_per_sentence.append(entities_in_sentence)

            if has_entity:
                entity_sentence_count += 1
            else:
                no_entity_sentence_count += 1

    # ====== REPORT ======
    print("\n===== SENTENCE-LEVEL EDA =====")
    print(f"Tổng số câu            : {total_sentences}")
    print(f"Số câu có entity       : {entity_sentence_count}")
    print(f"Số câu KHÔNG có entity : {no_entity_sentence_count}")

    print("\n===== ĐỘ DÀI CÂU (tokens) =====")
    print(f"Min  : {min(sentence_lengths)}")
    print(f"Max  : {max(sentence_lengths)}")
    print(f"Mean : {np.mean(sentence_lengths):.2f}")
    print(f"Median: {np.median(sentence_lengths)}")

    print("\n===== ENTITY PER SENTENCE =====")
    print(f"Trung bình entity / câu: {np.mean(entity_per_sentence):.2f}")
    print(f"Max entity trong 1 câu : {max(entity_per_sentence)}")

    print("\n===== PHÂN BỐ ENTITY TYPES =====")
    for ent, cnt in entity_counter.most_common():
        print(f"{ent:<10}: {cnt}")

    print("\n===== DỮ LIỆU LỖI =====")
    print(f"Số câu lỗi: {len(bad_sentences)}")
    if bad_sentences:
        print("Ví dụ lỗi đầu tiên:", bad_sentences[0])


In [10]:
sentence_level_eda(
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl"
)


===== SENTENCE-LEVEL EDA =====
Tổng số câu            : 36750
Số câu có entity       : 33212
Số câu KHÔNG có entity : 3538

===== ĐỘ DÀI CÂU (tokens) =====
Min  : 2
Max  : 120
Mean : 30.26
Median: 29.0

===== ENTITY PER SENTENCE =====
Trung bình entity / câu: 2.68
Max entity trong 1 câu : 26

===== PHÂN BỐ ENTITY TYPES =====
ORG       : 39631
DATE      : 20437
MONEY     : 13361
RATE      : 11932
ASSET     : 7044
PERSON    : 3500
VOLUME    : 2596

===== DỮ LIỆU LỖI =====
Số câu lỗi: 0


In [11]:
sentence_level_eda(
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/dev_vifinner.jsonl"
)


===== SENTENCE-LEVEL EDA =====
Tổng số câu            : 7890
Số câu có entity       : 7107
Số câu KHÔNG có entity : 783

===== ĐỘ DÀI CÂU (tokens) =====
Min  : 3
Max  : 100
Mean : 30.11
Median: 29.0

===== ENTITY PER SENTENCE =====
Trung bình entity / câu: 2.65
Max entity trong 1 câu : 24

===== PHÂN BỐ ENTITY TYPES =====
ORG       : 8378
DATE      : 4426
MONEY     : 2824
RATE      : 2579
ASSET     : 1398
PERSON    : 732
VOLUME    : 581

===== DỮ LIỆU LỖI =====
Số câu lỗi: 0


In [12]:
sentence_level_eda(
    "/Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/test_vifinner.jsonl"
)


===== SENTENCE-LEVEL EDA =====
Tổng số câu            : 7852
Số câu có entity       : 7103
Số câu KHÔNG có entity : 749

===== ĐỘ DÀI CÂU (tokens) =====
Min  : 2
Max  : 114
Mean : 30.25
Median: 29.0

===== ENTITY PER SENTENCE =====
Trung bình entity / câu: 2.69
Max entity trong 1 câu : 20

===== PHÂN BỐ ENTITY TYPES =====
ORG       : 8475
DATE      : 4399
MONEY     : 2892
RATE      : 2471
ASSET     : 1576
PERSON    : 725
VOLUME    : 570

===== DỮ LIỆU LỖI =====
Số câu lỗi: 0
